In [ ]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv

import subprocess
import uuid
from pathlib import Path
import os

In [ ]:
load_dotenv()

In [ ]:
llm = ChatOpenAI(
    model="meta-llama/llama-3.3-70b-instruct",
    # model="nvidia/nemotron-3.5-lightning:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [ ]:
class ChatState(TypedDict):
    user_message: str
    llm_response: str
    audio_file: str

In [ ]:
def chat_node(state: ChatState):

    user_message = state["user_message"]

    response = llm.invoke(user_message)

    return {
        "llm_response": response.content
    }

In [ ]:
AUDIO_DIR = Path("audio")
AUDIO_DIR.mkdir(exist_ok=True)

PIPER_EXECUTABLE = Path(
    ".venv/Scripts/piper.exe"
)

PIPER_MODEL = Path(
    "backend/models/en_US-lessac-medium.onnx"
)

In [ ]:
def piper_node(state: ChatState):

    text = state["llm_response"]

    filename = f"{uuid.uuid4()}.wav"

    output_path = AUDIO_DIR / filename

    command = [
        str(PIPER_EXECUTABLE),
        "--model",
        str(PIPER_MODEL),
        "--output_file",
        str(output_path),
    ]

    subprocess.run(
        command,
        input=text,
        text=True,
        check=True,
    )

    return {
        "audio_file": str(output_path)
    }

In [ ]:
builder = StateGraph(ChatState)

In [ ]:
builder.add_node(
    "chat",
    chat_node
)

builder.add_node(
    "piper",
    piper_node
)

In [ ]:
builder.add_edge(
    START,
    "chat"
)

builder.add_edge(
    "chat",
    "piper"
)

builder.add_edge(
    "piper",
    END
)

In [ ]:
graph = builder.compile()

In [ ]:
from IPython.display import Image, display

display(
    Image(
        graph.get_graph().draw_mermaid_png()
    )
)

In [ ]:
user_input = input("You: Tell me about yourself")

result = graph.invoke({
    "user_message": user_input,
})